# 02 — Train YOLO11-seg trên TuSimple, test trực tiếp trên Forza Horizon 4

**Luồng:** TuSimple (train/val, có label) → chuyển sang định dạng YOLO-seg → fine-tune `yolo11n-seg.pt` → đánh giá trên val → **test trực tiếp trên FH4** (chạy thời gian thực, view-only, không gửi lệnh điều khiển).

> ⚠️ FH4: chạy **offline**, góc nhìn **bumper cam**, cửa sổ windowed/borderless.


In [1]:
# ===== Cell 1: Setup & kiểm tra môi trường =====
import sys, subprocess
from pathlib import Path
ROOT_DIR = Path.cwd().parent
sys.path.insert(0, str(ROOT_DIR))

print('Project root:', ROOT_DIR)

from src.path import RAW_DATA_DIR, PROCESSED_DATA_DIR

try:
    from ultralytics import YOLO
    import torch
    print('ultralytics OK | device:', 'cuda' if torch.cuda.is_available() else 'cpu/mps')
except ImportError as e:
    print('Thiếu thư viện:', e)
    print('Cài: pip install ultralytics torch torchvision')


Project root: h:\FPT\4\CPV\project\CPV301



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.1.3 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "h:\FPT\4\CPV\project\CPV301\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "h:\FPT\4\CPV\project\CPV301\.venv\Lib\site-packages\traitlets\config\application.py", line 1080, in launch_instance
    app.start()
  File "h:\FPT\4\CPV\project\CPV301\.venv\Lib\site-packages\ipykernel\kernelapp.py", line 768, in start
    self.io_loop.start()
  File "h:\FPT\4\CP

ultralytics OK | device: cpu/mps


## Bước 1 — Chuẩn bị dữ liệu TuSimple

1. Tải **TuSimple train set** từ trang chính thức (hoặc nguồn nhóm đã chuẩn bị) và giải nén vào `data/external/tusimple/` sao cho có cấu trúc:
```
data/external/tusimple/
├── clips/0313-1/20/20.jpg  (và các clip 0531*, 0601*)
├── label_data_0313.json
├── label_data_0531.json
└── label_data_0601.json
```
2. Chạy cell dưới để chuyển label TuSimple (điểm làn) sang **mask polygon YOLO-seg**, chia train/val **theo clip** (tránh leak), xuất ra `data/processed/` (khớp `config/data.yaml`).


In [2]:
# ===== Cell 2: Chuyển đổi TuSimple -> YOLO-seg =====
TUSIMPLE_ROOT = RAW_DATA_DIR/ "TUSimple"

assert (TUSIMPLE_ROOT / 'test_set').is_dir(), f'Chưa thấy {TUSIMPLE_ROOT}/test_set — tải & giải nén TuSimple trước'

cmd = [
    sys.executable, str(ROOT_DIR / 'scripts' / 'tusimple_to_yolo.py'),
    '--tusimple_root', str(TUSIMPLE_ROOT),
    '--output_dir', str(PROCESSED_DATA_DIR),
    '--val_ratio', '0.1',
    '--lane_thickness', '12',
]
print('Chạy:', ' '.join(cmd))

result = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    encoding='utf-8',
    errors='replace'
)

print("Return code:", result.returncode)
print("\n===== STDOUT =====")
print(result.stdout)

print("\n===== STDERR =====")
print(result.stderr)


Chạy: h:\FPT\4\CPV\project\CPV301\.venv\Scripts\python.exe h:\FPT\4\CPV\project\CPV301\scripts\tusimple_to_yolo.py --tusimple_root h:\FPT\4\CPV\project\CPV301\data\raw\TUSimple --output_dir h:\FPT\4\CPV\project\CPV301\data\processed --val_ratio 0.1 --lane_thickness 12
Return code: 2

===== STDOUT =====


===== STDERR =====
C:\Users\pc\AppData\Local\Programs\Python\Python311\python.exe: can't open file 'h:\\FPT\\4\\CPV\\project\\CPV301\\scripts\\tusimple_to_yolo.py': [Errno 2] No such file or directory



## Bước 2 — Kiểm tra dữ liệu & cấu hình

`config/data.yaml` đã trỏ tới `../data/processed` với 1 class (`lane`). Cell dưới đếm số ảnh/label và hiển thị 1 mẫu để xác nhận mask đúng.


In [3]:
# ===== Cell 3: Kiểm tra dataset =====
import cv2, random
import numpy as np
import matplotlib.pyplot as plt

for split in ('train', 'valid'):
    imgs = list((PROCESSED_DATA_DIR / split / 'images').glob('*.jpg'))
    lbls = list((PROCESSED_DATA_DIR / split / 'labels').glob('*.txt'))
    print(f'{split:5s}: {len(imgs)} ảnh | {len(lbls)} label')

# Hiển thị 1 mẫu có mask overlay
sample = random.choice(list((PROCESSED_DATA_DIR / 'train' / 'images').glob('*.jpg')))
img = cv2.cvtColor(cv2.imread(str(sample)), cv2.COLOR_BGR2RGB)
h, w = img.shape[:2]
overlay = img.copy()
label_file = PROCESSED_DATA_DIR / 'train' / 'labels' / (sample.stem + '.txt')
for line in label_file.read_text().strip().splitlines():
    vals = list(map(float, line.split()))
    cls, pts = vals[0], vals[1:]
    poly = np.array([(int(pts[i]*w), int(pts[i+1]*h)) for i in range(0, len(pts), 2)], dtype=np.int32)
    if len(poly) >= 3:
        cv2.fillPoly(overlay, [poly], (0, 255, 0))
plt.figure(figsize=(10,5)); plt.imshow(cv2.addWeighted(img,0.6,overlay,0.4,0)); plt.axis('off'); plt.title(sample.name); plt.show()


train: 0 ảnh | 0 label
valid: 0 ảnh | 0 label


IndexError: Cannot choose from an empty sequence

## Bước 3 — Train YOLO11-seg

Fine-tune từ `yolo11n-seg.pt` (nano, nhanh). Tăng `epochs`/`imgsz` nếu có GPU mạnh. Kết quả lưu vào `runs/segment/train/`, weights tốt nhất là `weights/best.pt`.


In [4]:
# ===== Cell 4: Train =====
from ultralytics import YOLO

model = YOLO('yolo11n-seg.pt')   # auto-download lần đầu
results = model.train(
    data=str(ROOT_DIR / 'config' / 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,                     # 0 = GPU đầu tiên; 'cpu' nếu không có GPU
    project=str(ROOT_DIR / 'runs' / 'segment'),
    name='tusimple_lane',
    exist_ok=True,
    patience=10,                  # early stopping
)
print('Train xong. best.pt tại:', ROOT_DIR / 'runs' / 'segment' / 'tusimple_lane' / 'weights' / 'best.pt')


New https://pypi.org/project/ultralytics/8.4.183 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.182  Python-3.11.0 torch-2.1.2+cpu 


ValueError: Invalid CUDA 'device=0' requested. Use 'device=cpu' or pass valid CUDA device(s) if available, i.e. 'device=0' or 'device=0,1,2,3' for Multi-GPU.

torch.cuda.is_available(): False
torch.cuda.device_count(): 0
os.environ['CUDA_VISIBLE_DEVICES']: None
See https://pytorch.org/get-started/locally/ for up-to-date torch install instructions if no CUDA devices are seen by torch.


## Bước 4 — Đánh giá trên tập val + inference thử

Đo mAP50/mAP50-95 segmentation trên `valid`, rồi predict vài ảnh để mắt thường kiểm tra.


In [ ]:
# ===== Cell 5: Val + predict mẫu =====
BEST = ROOT_DIR / 'runs' / 'segment' / 'tusimple_lane' / 'weights' / 'best.pt'
assert BEST.exists(), f'Chưa thấy {BEST} — chạy cell Train trước'

model = YOLO(str(BEST))
metrics = model.val(data=str(ROOT_DIR / 'config' / 'data.yaml'), split='val')
print('mAP50-seg:', metrics.box.map50 if hasattr(metrics, 'box') else metrics)

# Predict 3 ảnh val
val_imgs = sorted((PROCESSED_DATA_DIR / 'valid' / 'images').glob('*.jpg'))[:3]
for p in val_imgs:
    r = model.predict(str(p), conf=0.5, verbose=False)[0]
    plt.figure(figsize=(10,5)); plt.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB)); plt.axis('off'); plt.show()


## Bước 5 — Test TRỰC TIẾP trên Forza Horizon 4 (view-only)

Không cần gán label FH4. Chạy script sau **trên Windows, FH4 đang chạy (offline, bumper cam)**: script chụp màn hình → detect → hiển thị cửa sổ overlay, **không gửi lệnh điều khiển**.

```bash
# Hough (không cần GPU, ~60-90 FPS)
python scripts/fh4_viewonly.py --detector hough --region 0 0 1280 720

# YOLO vừa train xong trên TuSimple (đây là cross-domain test: train TuSimple -> test FH4)
python scripts/fh4_viewonly.py --detector yolo \
    --weights runs/segment/tusimple_lane/weights/best.pt \
    --region 0 0 1280 720 --save data/output/fh4_viewonly_yolo.mp4
```

**Quan sát & ghi nhận** (vào báo cáo): spline có bám làn? FP (phát hiện nhầm lan can/bóng)? FN (mất làn ở khúc cua/đổi ánh sáng)? FPS thực tế? So sánh Hough vs YOLO-TuSimple trên FH4. Nếu YOLO tụt mạnh → đó là domain gap; bước tiếp theo là fine-tune thêm với ~500 ảnh FH4.


In [ ]:
# ===== Cell 6: In lệnh chạy test FH4 (chạy ngoài notebook, trên Windows) =====
BEST = ROOT_DIR / 'runs' / 'segment' / 'tusimple_lane' / 'weights' / 'best.pt'
print('Chạy các lệnh sau trên terminal Windows (FH4 đang chạy, offline, bumper cam):')
print('  python scripts/fh4_viewonly.py --detector hough --region 0 0 1280 720')
if BEST.exists():
    print(f'  python scripts/fh4_viewonly.py --detector yolo --weights {BEST} --region 0 0 1280 720 --save data/output/fh4_viewonly_yolo.mp4')
else:
    print('  (chưa có best.pt — train xong cell 4 thì lệnh YOLO sẽ hiện ở đây)')
